# Predicting Experimental Semiconductor Band Gaps from Chemical Formula

This notebook introduces a simple materials informatics workflow for predicting experimental semiconductor band gaps from chemical formula. The goal is not to build a perfect model. The goal is to learn how a chemical formula can be converted into numerical descriptors, used to train machine learning models, and interpreted critically.

We will use a public composition-only experimental band gap dataset from `matminer`, parse formulas with `pymatgen`, generate Magpie composition features with `matminer`, and compare a simple baseline model with a random forest model.

## How This Notebook Works

This notebook follows a full materials informatics workflow:

```text
chemical formula -> pymatgen Composition -> matminer features -> machine-learning model -> predicted band gap -> chemical interpretation
```

Each library has a specific job:

- `pandas` stores the dataset as tables so we can inspect rows and columns.
- `pymatgen` helps Python understand chemical compositions instead of treating formulas as plain text.
- `matminer` turns materials objects into numerical descriptors that a model can use.
- `scikit-learn` trains machine-learning models and evaluates their predictions.
- `matplotlib` makes figures that help us judge model behavior.

The scientific question is not only whether the model can predict band gaps. We also want to ask what the model learns, where it fails, and what chemistry is missing from formula-only data.

## What You Should Learn

By the end of this notebook, you should be able to:

- Explain why chemical formulas must be converted into numerical features before machine learning.
- Use `pymatgen` to parse formulas into composition objects.
- Use `matminer` to generate composition-based descriptors.
- Compare a baseline model, random forest model, and gradient boosting model.
- Interpret MAE, R2, parity plots, feature importance, and edge-case failures.
- Explain why composition-only models cannot fully predict real materials behavior.

## 0. Google Colab Setup

If you opened this notebook in Google Colab, run the next cell before continuing. It installs the packages that are usually missing from a fresh Colab session. If you are running locally from the repository environment, the cell will simply tell you that no Colab setup is needed.

In [ ]:
# This setup cell keeps the notebook runnable in Google Colab.
# It installs the same core packages listed in requirements.txt when Colab is detected.
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Google Colab detected. Installing teaching-module dependencies...")
    # In Colab, %pip installs packages into the active notebook runtime.
    %pip -q install matminer pymatgen scikit-learn pandas numpy matplotlib ipywidgets
    print("Installation complete. Continue with the next cell.")
else:
    print("Not running in Google Colab. Use requirements.txt for local setup.")

## 1. Import Required Packages

This section imports the Python libraries used in the notebook. The comments explain the role of each package.

In [ ]:
# pandas stores tabular data in DataFrames.
import pandas as pd

# numpy provides numerical tools and represents missing values as np.nan.
import numpy as np

# matplotlib creates the plots used to evaluate model predictions.
import matplotlib.pyplot as plt

# pathlib helps save figures using paths that work on different operating systems.
from pathlib import Path

# matminer provides public datasets and materials featurizers.
from matminer.datasets import load_dataset
from matminer.featurizers.composition import ElementProperty

# pymatgen understands chemical formulas and compositions.
from pymatgen.core import Composition

# scikit-learn provides train/test splitting, models, and metrics.
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# Show matplotlib plots inside the notebook.
%matplotlib inline

# Use a fixed random seed so the train/test split and random forest are reproducible.
RANDOM_STATE = 42

## 2. Load a Public Experimental Band Gap Dataset

The dataset is a table of experimental semiconductor band gaps. Each row corresponds to one material composition. The input variable is the formula or composition, and the target/output variable is the experimental band gap in electronvolts (eV).

This is a **supervised learning** problem because the model learns from examples with known answers: each composition is paired with an experimentally measured band gap.

The preferred dataset is `matbench_expt_gap`, a Matbench dataset for predicting experimental band gap from composition alone. If that dataset is not available in a particular `matminer` installation, the code tries closely related experimental gap datasets as fallbacks.

In [ ]:
# Try the preferred composition-only experimental band gap dataset first.
# Keeping a short fallback list makes the notebook more robust across matminer versions.
dataset_candidates = ["matbench_expt_gap", "expt_gap", "expt_gap_kingsbury"]

raw_data = None
dataset_name = None
load_errors = []

for candidate in dataset_candidates:
    try:
        # load_dataset returns a pandas DataFrame containing formulas/compositions and target values.
        raw_data = load_dataset(candidate)
        dataset_name = candidate
        break
    except Exception as exc:
        # Store failures so students can see what happened if no dataset loads.
        load_errors.append(f"{candidate}: {exc}")

if raw_data is None:
    raise RuntimeError("Could not load an experimental band gap dataset. Errors: " + " | ".join(load_errors))

print(f"Loaded dataset: {dataset_name}")
print(f"Number of rows: {len(raw_data):,}")

## 3. Inspect the First Rows and Column Names

Before modeling, always inspect the dataset. We need to identify which column contains the formula/composition and which column contains the experimental band gap target.

### Check your understanding

1. What does one row in this dataset represent?
2. Which column looks like the input variable?
3. Which column looks like the target variable?

In [ ]:
# Print column names so we can identify the input column and the target column.
print("Column names:")
print(list(raw_data.columns))

# Display the first few rows to see real examples before building a model.
raw_data.head()

## 4. Identify the Composition and Target Columns

Different datasets sometimes use different column names. The helper function below searches for likely formula/composition column names and likely band gap target column names. For `matbench_expt_gap`, the expected columns are usually `composition` and `gap expt`.

In [ ]:
def find_likely_column(columns, exact_candidates, keyword_groups, description):
    """Find a likely column using exact names first, then keyword groups."""
    normalized = {str(col).strip().lower(): col for col in columns}

    # Exact matches handle the common dataset column names first.
    for candidate in exact_candidates:
        key = candidate.strip().lower()
        if key in normalized:
            return normalized[key]

    # Keyword matches make the notebook more tolerant of small naming changes.
    for keywords in keyword_groups:
        for col in columns:
            col_text = str(col).strip().lower()
            if all(keyword in col_text for keyword in keywords):
                return col

    raise ValueError(f"Could not identify the {description} column. Available columns: {list(columns)}")


formula_col = find_likely_column(
    raw_data.columns,
    exact_candidates=["composition", "formula", "pretty_formula", "reduced_formula", "chemical_formula"],
    keyword_groups=[["composition"], ["formula"]],
    description="formula/composition",
)

target_col = find_likely_column(
    raw_data.columns,
    exact_candidates=["gap expt", "expt_gap", "band_gap", "band gap", "bandgap", "gap", "egap", "Egap"],
    keyword_groups=[["gap", "expt"], ["band", "gap"], ["gap"]],
    description="experimental band gap target",
)

print(f"Formula/composition column: {formula_col}")
print(f"Band gap target column: {target_col}")

# Show the two columns that define the supervised learning problem.
raw_data[[formula_col, target_col]].head()

## 5. Convert Formulas to pymatgen Composition Objects

### Concept: What does `pymatgen` do here?

A formula such as `CsPbBr3` is initially just text. `pymatgen` converts it into a `Composition` object that stores which elements are present and their ratios. For example, `CsPbBr3` contains 1 Cs, 1 Pb, and 3 Br atoms per formula unit.

A formula contains stoichiometry, but it does not contain crystal structure, polymorph, defects, particle size, surface chemistry, or synthesis history.

In [ ]:
def to_composition(value):
    """Convert a formula-like value to a pymatgen Composition object."""
    try:
        # Some matminer datasets already store pymatgen Composition objects.
        if isinstance(value, Composition):
            return value
        if value is None:
            return np.nan
        # For formula strings, Composition parses element symbols and stoichiometric ratios.
        return Composition(str(value))
    except Exception:
        # Returning np.nan lets us remove formulas that could not be parsed.
        return np.nan


# Keep only the input and target columns, then rename the target with clear units.
data = raw_data[[formula_col, target_col]].copy()
data = data.rename(columns={target_col: "band_gap_eV"})

# Convert target values to numbers. Non-numeric entries become missing values.
data["band_gap_eV"] = pd.to_numeric(data["band_gap_eV"], errors="coerce")

# Convert formulas or composition strings into pymatgen Composition objects.
data["composition"] = data[formula_col].apply(to_composition)

# Store a clean reduced formula string for display, plotting labels, and error analysis.
data["formula"] = data["composition"].apply(
    lambda comp: comp.reduced_formula if isinstance(comp, Composition) else np.nan
)

# Remove rows that do not have both a valid composition and a valid nonnegative band gap.
data = data.dropna(subset=["composition", "formula", "band_gap_eV"]).copy()
data = data[data["band_gap_eV"] >= 0].reset_index(drop=True)

print(f"Rows after formula parsing and target cleaning: {len(data):,}")
data[["formula", "composition", "band_gap_eV"]].head()

### Example Compositions

The examples below show how `pymatgen` interprets several familiar semiconductor or optoelectronic formulas.

In [ ]:
# These examples provide chemical anchors for the rest of the notebook.
example_formulas = ["Si", "GaAs", "CdTe", "PbS", "TiO2", "CsPbBr3"]

example_rows = []
for formula in example_formulas:
    comp = Composition(formula)
    example_rows.append(
        {
            "formula": formula,
            "reduced_formula": comp.reduced_formula,
            "elements": ", ".join(str(element) for element in comp.elements),
            "amounts": comp.get_el_amt_dict(),
        }
    )

# The table helps connect formula notation to the object that Python will use.
pd.DataFrame(example_rows)

In [ ]:
# Print the same examples in a more narrative format.
# This is useful when you want to read one composition at a time.
for formula in example_formulas:
    comp = Composition(formula)

    # reduced_formula gives the simplest integer formula that pymatgen recognizes.
    print(f"Formula: {formula}")
    print(f"  Reduced formula: {comp.reduced_formula}")

    # comp.elements lists the element objects present in the composition.
    print(f"  Elements: {[str(element) for element in comp.elements]}")

    # fractional_composition reports each element as a fraction of the total atoms.
    print(f"  Fractional composition: {comp.fractional_composition.as_dict()}")
    print()

### Check your understanding

1. What information is contained in a formula such as `TiO2`?
2. What information is missing from a formula such as `CsPbBr3`?
3. Why might two materials with the same composition have different measured band gaps?

## Understanding Magpie Features: Turning Chemistry into Numbers

Before we ask `matminer` to calculate features for the full dataset, it is worth slowing down to understand what a Magpie feature actually means.

Magpie features start with **elemental properties** from the periodic table or from elemental reference data. For each chemical formula, `matminer` combines the elemental values using simple statistics such as the minimum, maximum, range, mean, average deviation, and mode. The result is a numeric vector that a machine-learning model can read:

```text
formula -> pymatgen Composition -> Magpie elemental-property statistics -> numeric feature vector
```

For example, `PbS` contains Pb and S. A Magpie featurizer can summarize this composition using quantities such as mean electronegativity, range of electronegativity, mean atomic number, range of covalent radius, and mean valence-electron count.

These descriptors can be chemically useful because band gaps often depend on bonding, orbital energies, atomic size, ionicity, and periodic-table position. However, they are **not direct band gap calculations**. A Magpie feature does not know the crystal structure, orbital band dispersion, defects, grain size, synthesis route, or measurement method.


### Common Magpie elemental properties

The table below translates several Magpie property names into plain language. Some properties describe neutral elements in reference states rather than the actual compound structure, so treat them as useful clues rather than complete physical explanations.

| Magpie property | Plain-language meaning | Chemical intuition for band gaps |
|---|---|---|
| `Number` | Atomic number. | Tracks nuclear charge and often correlates with heavier atoms, stronger spin-orbit effects, and smaller gaps in many families. |
| `MendeleevNumber` | A one-number ordering of elements by periodic-table chemistry. | Helps the model recognize broad chemical similarity beyond simple atomic number. |
| `AtomicWeight` | Atomic mass. | Heavy elements can correlate with different orbital energies, bonding, and relativistic effects. |
| `MeltingT` | Elemental melting temperature. | Rough proxy for bond strength in elemental reference states; not the melting point of the compound. |
| `Column` | Periodic-table group. | Relates to valence electron patterns and common oxidation states. |
| `Row` | Periodic-table period. | Captures shell size and how diffuse valence orbitals may be. |
| `CovalentRadius` | Approximate bonding radius of an atom. | Larger radii can imply longer bonds and different orbital overlap. |
| `Electronegativity` | Tendency of an atom to attract electrons in a bond. | Electronegativity differences relate to ionicity, bond polarity, and band edge separation. |
| `NsValence` | Number of s valence electrons. | Helps identify valence-shell filling and bonding possibilities. |
| `NpValence` | Number of p valence electrons. | Important for many semiconductor valence and conduction band states. |
| `NdValence` | Number of d valence electrons. | Captures transition-metal character and possible localized d states. |
| `NfValence` | Number of f valence electrons. | Captures f-electron character in lanthanides and actinides. |
| `NValence` | Total valence electron count. | Broad descriptor of electron availability for bonding and band formation. |
| `NsUnfilled` | Number of unfilled s valence states. | Rough clue about available valence states. |
| `NpUnfilled` | Number of unfilled p valence states. | Can relate to electron accepting behavior and band-edge chemistry. |
| `NdUnfilled` | Number of unfilled d valence states. | May flag transition-metal compounds with low-lying d states. |
| `NfUnfilled` | Number of unfilled f valence states. | May flag f-electron materials with complex electronic behavior. |
| `NUnfilled` | Total number of unfilled valence states. | Broad descriptor of available valence-shell capacity. |
| `GSvolume_pa` | Ground-state elemental volume per atom. | Proxy for elemental size and packing; not the compound unit-cell volume. |
| `GSbandgap` | Band gap of the elemental reference state. | Can encode whether elements themselves are metallic, semiconducting, or insulating. |
| `GSmagmom` | Ground-state elemental magnetic moment. | May help identify elements associated with magnetic or open-shell behavior. |
| `SpaceGroupNumber` | Space group number of the elemental ground-state structure. | Encodes elemental structural tendencies, not the compound crystal structure. |


### Common Magpie statistics

For a compound, Magpie usually calculates several statistics for each elemental property. These statistics are weighted by stoichiometry when appropriate. For `TiO2`, oxygen contributes twice as many atoms as titanium, so the mean electronegativity is pulled more strongly toward oxygen than toward titanium.

| Statistic | Meaning | Example interpretation |
|---|---|---|
| `minimum` | Smallest elemental value among the elements in the composition. | In `PbS`, the lower electronegativity belongs to Pb. |
| `maximum` | Largest elemental value among the elements in the composition. | In `TiO2`, the higher electronegativity belongs to O. |
| `range` | Difference between maximum and minimum. | A large electronegativity range can suggest more polar or ionic bonding. |
| `mean` | Stoichiometry-weighted average value. | In `TiO2`, the mean property value counts two O atoms for every one Ti atom. |
| `avg_dev` | Average absolute deviation from the mean. | Larger values mean the elements differ more strongly in that property. |
| `mode` | Most common elemental value by stoichiometric amount. | In `TiO2`, the mode of many properties corresponds to O because O appears twice. |


### Inspect the Magpie feature labels

Each Magpie feature name combines a statistic and an elemental property. For example, `MagpieData mean Electronegativity` means: calculate the stoichiometry-weighted mean electronegativity of the elements in the formula.


In [ ]:
# Create a Magpie featurizer so we can inspect its feature labels before using it on the full dataset.
magpie_featurizer = ElementProperty.from_preset("magpie")

# Single-process featurization is often easier to debug in teaching notebooks.
try:
    magpie_featurizer.set_n_jobs(1)
except AttributeError:
    pass

# The labels show exactly which numerical descriptors will be generated.
magpie_feature_names = magpie_featurizer.feature_labels()
print(f"Number of Magpie features: {len(magpie_feature_names)}")

# Display the first 20 labels as examples.
magpie_feature_names[:20]


### Explain a Magpie feature name

The helper below turns a technical Magpie feature label into a short chemical explanation. This is intentionally simple and transparent, so you can edit the dictionaries if you want different wording.


In [ ]:
MAGPIE_STATISTIC_MEANINGS = {
    "minimum": "the smallest elemental value among the elements in the composition",
    "maximum": "the largest elemental value among the elements in the composition",
    "range": "the difference between the largest and smallest elemental values",
    "mean": "the stoichiometry-weighted average elemental value",
    "avg_dev": "the average absolute deviation from the stoichiometry-weighted mean",
    "mode": "the elemental value associated with the most abundant element in the formula",
}

MAGPIE_PROPERTY_MEANINGS = {
    "Number": "atomic number",
    "MendeleevNumber": "a periodic-table ordering based on chemical similarity",
    "AtomicWeight": "atomic mass of the element",
    "MeltingT": "melting temperature of the elemental reference state",
    "Column": "periodic-table group number",
    "Row": "periodic-table period number",
    "CovalentRadius": "approximate covalent bonding radius",
    "Electronegativity": "tendency of an atom to attract electrons in a bond",
    "NsValence": "number of s valence electrons",
    "NpValence": "number of p valence electrons",
    "NdValence": "number of d valence electrons",
    "NfValence": "number of f valence electrons",
    "NValence": "total number of valence electrons",
    "NsUnfilled": "number of unfilled s valence states",
    "NpUnfilled": "number of unfilled p valence states",
    "NdUnfilled": "number of unfilled d valence states",
    "NfUnfilled": "number of unfilled f valence states",
    "NUnfilled": "total number of unfilled valence states",
    "GSvolume_pa": "ground-state elemental volume per atom",
    "GSbandgap": "band gap of the elemental reference state",
    "GSmagmom": "ground-state elemental magnetic moment",
    "SpaceGroupNumber": "space group number of the elemental ground-state structure",
}

MAGPIE_CHEMICAL_INTUITION = {
    "Number": "Heavy elements can correlate with smaller gaps, stronger spin-orbit effects, and different orbital energies.",
    "MendeleevNumber": "Elements near each other in chemical ordering often show related bonding or oxidation-state patterns.",
    "AtomicWeight": "Large atomic weight often travels with heavier, more polarizable atoms and different band-edge chemistry.",
    "MeltingT": "Elemental melting temperature is a rough proxy for bonding strength in the elemental reference state.",
    "Column": "Group number reflects valence-electron patterns that influence bonding and band filling.",
    "Row": "Period number reflects shell size and orbital diffuseness.",
    "CovalentRadius": "Bond length and orbital overlap often change with atomic radius.",
    "Electronegativity": "Electronegativity differences can indicate bond polarity, ionicity, and separation of band-edge states.",
    "NsValence": "s electrons can contribute to conduction or bonding states in many semiconductors.",
    "NpValence": "p electrons are central to many main-group semiconductor valence and conduction bands.",
    "NdValence": "d electrons can introduce transition-metal states that change the gap.",
    "NfValence": "f electrons can create localized states and complex electronic behavior.",
    "NValence": "Total valence electron count summarizes broad bonding and electron-filling tendencies.",
    "NsUnfilled": "Unfilled s states can signal available low-lying valence capacity.",
    "NpUnfilled": "Unfilled p states can affect electron accepting behavior and band-edge composition.",
    "NdUnfilled": "Unfilled d states can matter in transition-metal compounds.",
    "NfUnfilled": "Unfilled f states can matter in lanthanide and actinide compounds.",
    "NUnfilled": "Total unfilled states provide a broad clue about available valence-shell capacity.",
    "GSvolume_pa": "Elemental volume per atom is a proxy for atomic size and packing tendencies.",
    "GSbandgap": "Elemental reference band gaps can help distinguish elements that are metallic, semiconducting, or insulating alone.",
    "GSmagmom": "Magnetic moments can flag open-shell elements, but do not prove the compound is magnetic.",
    "SpaceGroupNumber": "Elemental structure can encode periodic trends, but it is not the compound structure.",
}


def explain_magpie_feature_name(feature_name):
    """Return a plain-language explanation of a Magpie feature label."""
    cleaned_name = str(feature_name).replace("MagpieData ", "", 1)
    parts = cleaned_name.split(maxsplit=1)

    if len(parts) == 2:
        statistic, elemental_property = parts
    else:
        statistic, elemental_property = cleaned_name, "unknown"

    return {
        "original feature name": str(feature_name),
        "statistic": statistic,
        "elemental property": elemental_property,
        "plain-language statistic meaning": MAGPIE_STATISTIC_MEANINGS.get(
            statistic,
            "statistic not recognized by this simple teaching helper",
        ),
        "property meaning": MAGPIE_PROPERTY_MEANINGS.get(
            elemental_property,
            "property not recognized by this simple teaching helper",
        ),
        "chemical intuition note": MAGPIE_CHEMICAL_INTUITION.get(
            elemental_property,
            "Use chemical judgment and inspect the feature carefully.",
        ),
    }


# Try the helper on a feature that often appears in band gap models.
explain_magpie_feature_name("MagpieData mean Electronegativity")


### Magpie Feature Explorer

Use this optional explorer to connect a formula, a descriptor name, and a numerical value. Choose a formula, choose a Magpie feature, and read the interpretation. If widgets do not render in your environment, call `explore_magpie_feature("PbS", "MagpieData mean Electronegativity")` manually.


In [ ]:
# The explorer uses widgets when available, but the notebook still runs without them.
try:
    import ipywidgets as widgets
    from ipywidgets import interact_manual
    from IPython.display import display
    MAGPIE_WIDGETS_AVAILABLE = True
except Exception as widget_error:
    MAGPIE_WIDGETS_AVAILABLE = False
    print("ipywidgets could not be imported. You can still call explore_magpie_feature(...) manually.")
    print(widget_error)

magpie_example_formulas = ["Si", "GaAs", "CdTe", "PbS", "TiO2", "ZnO", "Al2O3", "CsPbBr3"]
default_magpie_feature = (
    "MagpieData mean Electronegativity"
    if "MagpieData mean Electronegativity" in magpie_feature_names
    else magpie_feature_names[0]
)


def featurize_single_formula_for_magpie(formula):
    """Parse one formula and return a one-row table of Magpie features."""
    composition = Composition(str(formula).strip())
    single_formula_data = pd.DataFrame(
        {"formula": [composition.reduced_formula], "composition": [composition]}
    )
    return magpie_featurizer.featurize_dataframe(
        single_formula_data,
        col_id="composition",
        ignore_errors=True,
    )


def explore_magpie_feature(formula="PbS", feature_name=default_magpie_feature):
    """Show one Magpie feature value and a plain-language interpretation."""
    formula = str(formula).strip()
    if not formula:
        print("Please enter a chemical formula such as PbS or TiO2.")
        return None

    if feature_name not in magpie_feature_names:
        print(f"Feature '{feature_name}' was not found in the Magpie preset.")
        print("Try one of these examples:")
        print(magpie_feature_names[:10])
        return None

    try:
        feature_table = featurize_single_formula_for_magpie(formula)
    except Exception as exc:
        print(f"Could not parse or featurize formula '{formula}'. Check capitalization and stoichiometry.")
        print(exc)
        return None

    composition = feature_table.loc[0, "composition"]
    value = feature_table.loc[0, feature_name]
    explanation = explain_magpie_feature_name(feature_name)

    print(f"Formula: {formula}")
    print(f"Reduced formula: {composition.reduced_formula}")
    print(f"Feature: {feature_name}")
    print(f"Numerical value: {value:.4g}" if pd.notna(value) else "Numerical value: missing")
    print()
    print("Interpretation:")
    for key, text in explanation.items():
        print(f"- {key}: {text}")

    return pd.DataFrame([explanation])


# Show one non-interactive example so automated notebook execution produces visible output.
example_magpie_explanation = explore_magpie_feature("PbS", default_magpie_feature)

if MAGPIE_WIDGETS_AVAILABLE:
    interact_manual(
        explore_magpie_feature,
        formula=widgets.Combobox(
            value="PbS",
            options=magpie_example_formulas,
            description="Formula",
            ensure_option=False,
        ),
        feature_name=widgets.Dropdown(
            options=magpie_feature_names,
            value=default_magpie_feature,
            description="Feature",
            layout=widgets.Layout(width="650px"),
            style={"description_width": "initial"},
        ),
    )


### Compare selected descriptors across familiar formulas

The next table calculates a small subset of descriptors for familiar semiconductors and insulators. Use it to compare chemistry before looking at model performance.


In [ ]:
def normalize_magpie_label(label):
    """Normalize a feature label so matching is robust to spaces and punctuation."""
    return "".join(character for character in str(label).lower() if character.isalnum())


def find_magpie_feature(statistic, elemental_property, available_features):
    """Find an exact Magpie feature label, with a simple fallback match."""
    target = normalize_magpie_label(f"MagpieData {statistic} {elemental_property}")
    normalized_features = {normalize_magpie_label(feature): feature for feature in available_features}

    if target in normalized_features:
        return normalized_features[target]

    statistic_token = normalize_magpie_label(statistic)
    property_token = normalize_magpie_label(elemental_property)
    for feature in available_features:
        normalized = normalize_magpie_label(feature)
        if statistic_token in normalized and property_token in normalized:
            return feature

    return None


selected_descriptor_requests = [
    ("mean", "Electronegativity"),
    ("range", "Electronegativity"),
    ("mean", "AtomicWeight"),
    ("range", "AtomicWeight"),
    ("mean", "CovalentRadius"),
    ("range", "CovalentRadius"),
    ("mean", "NValence"),
    ("avg_dev", "NValence"),
]

selected_magpie_features = [
    find_magpie_feature(statistic, elemental_property, magpie_feature_names)
    for statistic, elemental_property in selected_descriptor_requests
]
selected_magpie_features = [feature for feature in selected_magpie_features if feature is not None]

comparison_data = pd.DataFrame(
    {
        "formula": magpie_example_formulas,
        "composition": [Composition(formula) for formula in magpie_example_formulas],
    }
)
comparison_features = magpie_featurizer.featurize_dataframe(
    comparison_data,
    col_id="composition",
    ignore_errors=True,
)

# Keep the display focused on a few descriptors students can reason about chemically.
comparison_columns = ["formula"] + selected_magpie_features
comparison_features[comparison_columns].round(3)


### Descriptor intuition checkpoint

1. Which example formula has the largest electronegativity range? Does that match your expectation about ionic or polar bonding?
2. Compare `PbS`, `CdTe`, and `GaAs`. How do their atomic-weight and covalent-radius descriptors differ?
3. Compare `TiO2`, `ZnO`, and `Al2O3`. What do the electronegativity and valence-electron descriptors suggest about oxide chemistry?
4. Choose one descriptor from the table. Explain why it might help predict a band gap, and also why it is not a complete explanation by itself.
5. Which missing information would you want next if two formulas had similar Magpie descriptors but very different measured band gaps?


## 6. Create Magpie Composition Features with matminer

### Concept: What is featurization?

Machine-learning models cannot directly use chemical formulas like `PbS` or `TiO2`. They need numbers. Featurization converts each composition into numerical descriptors such as average electronegativity, range of atomic numbers, mean atomic radius, and valence-electron statistics.

In this notebook, `matminer` uses Magpie-style elemental descriptors. These features summarize periodic-table information for the elements in each formula.

In [ ]:
# Use the full dataset by default.
# For a faster classroom demo, set MAX_ROWS to 1500 or another smaller number.
MAX_ROWS = None

model_data = data[["formula", "composition", "band_gap_eV"]].copy()
if MAX_ROWS is not None and MAX_ROWS < len(model_data):
    # Sampling keeps the workflow fast while preserving the same code path.
    model_data = model_data.sample(n=MAX_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Rows selected for featurization: {len(model_data):,}")

In [ ]:
# ElementProperty.from_preset("magpie") creates many composition descriptors
# based on elemental properties from the Magpie descriptor set.
featurizer = ElementProperty.from_preset("magpie")

# Single-process featurization is often easier to debug on teaching machines.
try:
    featurizer.set_n_jobs(1)
except AttributeError:
    pass

# Featurization turns each Composition into a row of numerical descriptors.
# Some feature names look long or technical because they encode both the elemental
# property and the statistic being calculated, such as a mean, range, or maximum.
# ignore_errors=True keeps the notebook running if an unusual composition fails.
featurized = featurizer.featurize_dataframe(
    model_data,
    col_id="composition",
    ignore_errors=True,
)

print(f"Shape after featurization: {featurized.shape}")
featurized.head()

## 7. Drop Failed Featurizations and Missing Values

Featurization can produce missing values if an elemental property is unavailable or if a composition cannot be processed cleanly. For this beginner workflow, we remove rows with missing target values or missing numerical features so the models receive a clean table.

Feature names can look technical because each one describes a statistic of an elemental property. For example, a feature might represent the range of an elemental property across all elements in the formula.

### Check your understanding

1. Why does the model need numerical features instead of formula strings?
2. Why might a feature value be missing?
3. What is the tradeoff of dropping rows with missing values?

In [ ]:
# Metadata columns describe the material or target; they are not input features.
metadata_cols = {"formula", "composition", "band_gap_eV"}

# Select only numeric descriptor columns created by the Magpie featurizer.
feature_cols = [
    col
    for col in featurized.columns
    if col not in metadata_cols and pd.api.types.is_numeric_dtype(featurized[col])
]

# Replace infinite values with missing values so they can be removed consistently.
featurized[feature_cols] = featurized[feature_cols].replace([np.inf, -np.inf], np.nan)

# Drop rows with missing target values or missing feature values.
# This keeps the first modeling workflow simple for students.
rows_before = len(featurized)
clean_data = featurized.dropna(subset=["band_gap_eV"] + feature_cols).reset_index(drop=True)
rows_after = len(clean_data)

print(f"Rows before dropping missing values: {rows_before:,}")
print(f"Rows after dropping missing values: {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Number of Magpie features: {len(feature_cols)}")

# Display a few feature columns so students can see what the model receives.
clean_data[["formula", "band_gap_eV"] + feature_cols[:5]].head()

## 8. Split into Train and Test Data

### Concept: Why split into training and test data?

The model learns from the training set. We evaluate it on the test set, which contains materials the model did not see during training. This helps us estimate whether the model can generalize rather than simply memorize.

In [ ]:
# X contains numerical composition features. y contains experimental band gaps.
X = clean_data[feature_cols]
y = clean_data["band_gap_eV"]

# Keep formulas and Composition objects so later error analysis can connect predictions back to chemistry.
metadata = clean_data[["formula", "composition"]]

# Split the data so model performance is evaluated on materials it did not see during training.
X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X,
    y,
    metadata,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

## 9. Train a DummyRegressor Baseline

### Model 1: Baseline model

The baseline model is intentionally simple. It predicts the average band gap for every material. A useful machine-learning model should perform better than this simple reference.

### Check your understanding

1. Why should we compare against a baseline model?
2. What information does the baseline ignore?

In [ ]:
# The baseline gives us a minimum standard for model performance.
# It ignores chemical features and always predicts the training-set mean band gap.
baseline_model = DummyRegressor(strategy="mean")

# .fit() lets the baseline learn the mean of y_train.
baseline_model.fit(X_train, y_train)

# .predict() applies that learned mean to every material in the test set.
baseline_pred = baseline_model.predict(X_test)

print("Baseline model trained.")

## 10. Train a RandomForestRegressor

### Model 2: Random forest

A random forest is an ensemble of decision trees. Each tree learns a set of rules based on the features, and the forest averages many trees to make a prediction. Random forests are useful here because they can learn nonlinear relationships between composition-derived features and band gap.

In [ ]:
# Train a random forest model on the same training data used by the baseline.
rf_model = RandomForestRegressor(
    # n_estimators is the number of decision trees in the forest.
    n_estimators=300,
    # random_state makes the model reproducible for teaching and debugging.
    random_state=RANDOM_STATE,
    # n_jobs=-1 lets scikit-learn use available CPU cores for faster training.
    n_jobs=-1,
    # max_features="sqrt" lets each tree consider a subset of features at each split.
    max_features="sqrt",
)

# .fit() trains the forest by finding patterns between features and known band gaps.
rf_model.fit(X_train, y_train)

# .predict() estimates band gaps for test-set materials the model did not train on.
rf_pred = rf_model.predict(X_test)

print("Random forest model trained.")

## Random Forest Results: MAE and R2

### How do we evaluate the model?

Mean absolute error, or MAE, is the average absolute difference between predicted and actual band gap. Because band gap is measured in eV, MAE is also reported in eV.

R2 measures how much variation in the target values is explained by the model. An R2 of 1 is perfect. An R2 near 0 means the model is not much better than predicting the average. A negative R2 means the model is worse than the average-prediction baseline.

In [ ]:
def regression_metrics(y_true, y_pred, model_name):
    """Calculate common regression metrics for a model."""
    return {
        "model": model_name,
        "MAE_eV": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


# Compare the baseline and random forest on the same held-out test set.
metrics = pd.DataFrame(
    [
        regression_metrics(y_test, baseline_pred, "DummyRegressor baseline"),
        regression_metrics(y_test, rf_pred, "RandomForestRegressor"),
    ]
)

# Rounding makes the table easier to read in a classroom discussion.
metrics.round({"MAE_eV": 3, "R2": 3})

## Random Forest Diagnostic Plot: Predicted vs. Actual Band Gap

### How to read the parity plot

Each point is one material in the test set. The x-axis is the experimental band gap, and the y-axis is the model-predicted band gap. Perfect predictions would fall on the diagonal line. Points far from the line are model errors.

### Check your understanding

1. What does a point above the diagonal line mean?
2. What does a point below the diagonal line mean?
3. Why might high-band-gap materials be harder to predict?

In [ ]:
# Save figures in the project-level figures directory when possible.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
figure_dir = project_root / "figures"
figure_dir.mkdir(exist_ok=True)

# Create a parity plot to compare predicted and experimental values directly.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, rf_pred, alpha=0.65, s=35, color="#4C78A8", edgecolor="white", linewidth=0.4)

# The diagonal line marks perfect prediction.
min_gap = min(y_test.min(), rf_pred.min())
max_gap = max(y_test.max(), rf_pred.max())
ax.plot([min_gap, max_gap], [min_gap, max_gap], color="#F58518", linestyle="--", label="Perfect prediction")

ax.set_xlabel("Experimental band gap (eV)")
ax.set_ylabel("Predicted band gap (eV)")
ax.set_title("Random forest: predicted vs. actual band gaps")
ax.legend()
ax.set_aspect("equal", adjustable="box")

pred_plot_path = figure_dir / "predicted_vs_actual_bandgaps.png"
fig.savefig(pred_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {pred_plot_path}")

## Built-In Random Forest Feature Importances

### How to interpret feature importance

Feature importance tells us which numerical descriptors the model relied on most. These features may connect to chemical ideas such as electronegativity, atomic size, valence electrons, or periodic-table position. However, feature importance is not proof of causation. A feature can be useful for prediction because it correlates with other physical information.

In [ ]:
# Pair each feature name with its built-in random forest importance value.
importance_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance": rf_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

# Plot the top 15 features so the figure is readable.
top15 = importance_df.head(15).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top15["feature"], top15["importance"], color="#54A24B")
ax.set_xlabel("Built-in random forest feature importance")
ax.set_title("Top 15 built-in random forest feature importances")

importance_plot_path = figure_dir / "top15_feature_importances.png"
fig.savefig(importance_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {importance_plot_path}")
importance_df.head(15)

## 11. Edge-Case Clinic: Where Does the Model Fail?

### Why study the worst predictions?

The largest errors are not just failures. They are teaching examples. They show where composition-only descriptors are missing important physical information such as crystal structure, polymorph, disorder, defects, synthesis history, or measurement uncertainty.

In [ ]:
# Build an error table so we can connect numerical errors back to chemical formulas.
error_table = meta_test.copy().reset_index(drop=True)
error_table["actual_gap_eV"] = y_test.to_numpy()
error_table["predicted_gap_eV"] = rf_pred

# Signed error tells us direction: positive means overprediction, negative means underprediction.
error_table["signed_error_eV"] = error_table["predicted_gap_eV"] - error_table["actual_gap_eV"]

# Absolute error tells us magnitude regardless of direction.
error_table["absolute_error_eV"] = error_table["signed_error_eV"].abs()

# The largest absolute errors show where the model struggled most.
largest_errors = error_table.sort_values("absolute_error_eV", ascending=False)
largest_errors[["formula", "actual_gap_eV", "predicted_gap_eV", "absolute_error_eV", "signed_error_eV"]].head(15).round(3)

### Edge-Case Discussion Questions

1. Are the largest errors mostly high-gap or low-gap materials?
2. Are the errors mostly overpredictions or underpredictions?
3. What chemical or structural information is missing from formula-only descriptors?
4. How might crystal structure, polymorph, defects, synthesis history, or measurement uncertainty affect these examples?

### Chemical Family Labels

The next helper function assigns a simple chemical family label using the elements present in each `pymatgen` `Composition`. This is intentionally simple and transparent. It is not a complete chemical taxonomy.

In [ ]:
def simple_chemical_family(composition):
    """Assign a transparent, formula-only chemical family label."""
    comp = composition if isinstance(composition, Composition) else Composition(str(composition))
    elements = {str(element) for element in comp.elements}

    # Elemental materials contain only one element, such as Si.
    if len(elements) == 1:
        return "elemental"

    # The order matters: oxides are separated before broader chalcogenides.
    if "O" in elements:
        return "oxide"

    # Halides contain at least one halogen.
    halogens = {"F", "Cl", "Br", "I"}
    if elements & halogens:
        return "halide"

    # Chalcogenides here include S, Se, or Te compounds that were not oxides.
    chalcogens_without_oxygen = {"S", "Se", "Te"}
    if elements & chalcogens_without_oxygen:
        return "chalcogenide"

    # Pnictides include common group 15 elements in semiconductors such as As and Sb.
    pnictogens = {"N", "P", "As", "Sb", "Bi"}
    if elements & pnictogens:
        return "pnictide"

    return "other"


# Add a chemical family label to each test-set prediction.
error_table["chemical_family"] = error_table["composition"].apply(simple_chemical_family)

error_table[["formula", "chemical_family", "actual_gap_eV", "predicted_gap_eV", "absolute_error_eV"]].head()

### Mean Absolute Error by Chemical Family

This plot asks whether the random forest performs equally well for different broad chemical families. Differences can suggest where the training data, descriptors, or model are less reliable.

In [ ]:
# Calculate the random forest MAE separately for each simple chemical family.
family_mae = (
    error_table.groupby("chemical_family")
    .agg(
        mean_absolute_error_eV=("absolute_error_eV", "mean"),
        count=("absolute_error_eV", "size"),
    )
    .sort_values("mean_absolute_error_eV", ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(family_mae.index, family_mae["mean_absolute_error_eV"], color="#B279A2")
ax.set_xlabel("Chemical family")
ax.set_ylabel("Mean absolute error (eV)")
ax.set_title("Random forest error by simple chemical family")
ax.tick_params(axis="x", rotation=30)

family_plot_path = figure_dir / "mae_by_chemical_family.png"
fig.savefig(family_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {family_plot_path}")
family_mae.round(3)

## 12. Can a More Flexible Model Improve the Predictions?

### Model 3: Gradient boosting

Gradient boosting is another tree-based ensemble method. Instead of building many independent trees and averaging them like a random forest, boosting builds trees sequentially, where each new tree tries to improve on the mistakes of the previous trees. This can improve performance, but a more flexible model is not automatically more physically meaningful.

In [ ]:
# HistGradientBoostingRegressor is included in scikit-learn, so no extra package is needed.
from sklearn.ensemble import HistGradientBoostingRegressor

hgb_model = HistGradientBoostingRegressor(
    # max_iter controls how many boosting steps are used.
    max_iter=300,
    # learning_rate controls how strongly each new tree changes the model.
    learning_rate=0.05,
    # l2_regularization discourages overly complex fits.
    l2_regularization=0.01,
    random_state=RANDOM_STATE,
)

# Train and test the boosting model on the same split used above.
hgb_model.fit(X_train, y_train)
hgb_pred = hgb_model.predict(X_test)

# Compare all three models on the same test set so the comparison is fair.
model_comparison = pd.DataFrame(
    [
        regression_metrics(y_test, baseline_pred, "DummyRegressor baseline"),
        regression_metrics(y_test, rf_pred, "RandomForestRegressor"),
        regression_metrics(y_test, hgb_pred, "HistGradientBoostingRegressor"),
    ]
)

model_comparison.round({"MAE_eV": 3, "R2": 3})

### Model-Comparison Plot

The bar plot below compares test-set MAE for the baseline, random forest, and gradient boosting models. Lower MAE is better.

### Check your understanding

1. Did the more flexible model improve MAE?
2. Does a lower MAE automatically mean the model is more chemically meaningful?
3. Why is it important that all three models use the same train/test split?

In [ ]:
# A bar plot makes it easy to compare test errors across models.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(model_comparison["model"], model_comparison["MAE_eV"], color=["#9D755D", "#4C78A8", "#72B7B2"])
ax.set_ylabel("Test MAE (eV)")
ax.set_title("Model comparison: lower MAE is better")
ax.tick_params(axis="x", rotation=20)

comparison_plot_path = figure_dir / "model_comparison_mae.png"
fig.savefig(comparison_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {comparison_plot_path}")

## Interactive Challenge: Can Your Chemistry Intuition Improve the Model?

In the previous sections, we trained models using all available composition-derived features. In this challenge, you will decide which chemical information the model is allowed to use.

Before running the interactive cell, write down a hypothesis. For example:

- "I think electronegativity and valence-electron features will matter most for band gap."
- "I think oxides will need different information than chalcogenides."
- "I think a smaller set of chemically meaningful features may perform almost as well as all features."

Your goal is not only to lower the error. Your goal is to explain why a choice helped, failed, or helped only for certain chemical families.


Use the Magpie Feature Explorer above before choosing feature groups. Your feature choices should be based on a chemical hypothesis, not random trial and error.


### Challenge rules

- Keep the train/test split fixed.
- Do not use the band gap target as a feature.
- Do not tune only to one lucky run.
- A lower MAE is better, but inspect which materials improved or worsened.
- A model that is more accurate is not automatically more physically meaningful.
- A feature that is important for prediction is not automatically a direct physical cause.

### Automatic feature selection

Automatic feature selection chooses features based on statistical relationships with the target. This provides a comparison to chemistry-guided feature selection. If automatic feature selection performs better, students should inspect whether the selected features make chemical sense.

### Feature budget challenge

Try to beat the all-feature random forest using fewer features. Choose a small number of chemically meaningful features or feature groups. Record whether your reduced-feature model performs similarly, better, or worse.

### Chemical-family challenge

Choose one family of materials, such as oxides or chalcogenides. Try to improve the error for that family. Then check whether your choices improved the overall model or only helped that family.

In [ ]:
# The interactive challenge uses widgets when they are available.
# The notebook still executes from top to bottom if widgets do not render.
from datetime import datetime
from sklearn.feature_selection import SelectKBest, mutual_info_regression

try:
    import ipywidgets as widgets
    from ipywidgets import interact_manual
    from IPython.display import display
    WIDGETS_AVAILABLE = True
except Exception as widget_error:
    WIDGETS_AVAILABLE = False
    print("ipywidgets could not be imported. You can still call run_interactive_challenge(...) manually.")
    print(widget_error)

FEATURE_GROUP_KEYWORDS = {
    "Electronegativity": ["electronegativity"],
    "Atomic size / radius": ["radius", "covalentradius", "atomicradius"],
    "Atomic number / mass": ["atomicnumber", "atomicweight", "number"],
    "Periodic table row / column": ["column", "row"],
    "Valence electrons": ["nvalence", "nsvalence", "npvalence", "ndvalence", "nfvalence"],
    "Thermal / melting": ["melting"],
    "All features": [],
}


def normalize_feature_name(name):
    """Normalize feature names so keyword matching is robust to spaces and punctuation."""
    return "".join(character for character in str(name).lower() if character.isalnum())


def get_feature_columns_for_groups(selected_groups, all_feature_columns):
    """Return feature columns matching one or more chemistry-inspired feature groups."""
    selected_groups = list(selected_groups) if selected_groups is not None else []

    if "All features" in selected_groups:
        return list(all_feature_columns)

    selected_columns = []
    missing_groups = []
    normalized_columns = {column: normalize_feature_name(column) for column in all_feature_columns}

    for group in selected_groups:
        keywords = FEATURE_GROUP_KEYWORDS.get(group, [])
        normalized_keywords = [normalize_feature_name(keyword) for keyword in keywords]
        group_columns = [
            column
            for column, normalized_column in normalized_columns.items()
            if any(keyword in normalized_column for keyword in normalized_keywords)
        ]

        if not group_columns:
            print(f"Warning: feature group '{group}' matched zero columns.")
            missing_groups.append(group)

        selected_columns.extend(group_columns)

    if missing_groups:
        print("Skipping training because at least one selected feature group had no matching columns.")
        return []

    # Remove duplicates while preserving order.
    return list(dict.fromkeys(selected_columns))


def select_automatic_top_k_features(k_best, all_feature_columns):
    """Select top-k features using mutual information fitted only on the training set."""
    requested_k = int(k_best)
    actual_k = min(requested_k, len(all_feature_columns))

    if requested_k > len(all_feature_columns):
        print(f"Requested k={requested_k}, but only {len(all_feature_columns)} features are available. Using k={actual_k}.")

    selector = SelectKBest(
        score_func=lambda X_values, y_values: mutual_info_regression(
            X_values,
            y_values,
            random_state=RANDOM_STATE,
        ),
        k=actual_k,
    )
    selector.fit(X_train[all_feature_columns], y_train)

    selected_mask = selector.get_support()
    return list(np.array(all_feature_columns)[selected_mask])


def family_mae_from_predictions(predictions):
    """Calculate MAE by chemical family for a set of test-set predictions."""
    family_table = meta_test.copy().reset_index(drop=True)
    family_table["actual_gap_eV"] = y_test.to_numpy()
    family_table["predicted_gap_eV"] = predictions
    family_table["chemical_family"] = family_table["composition"].apply(simple_chemical_family)
    family_table["absolute_error_eV"] = (family_table["predicted_gap_eV"] - family_table["actual_gap_eV"]).abs()
    return family_table.groupby("chemical_family")["absolute_error_eV"].mean().sort_values()


# Reference results from earlier all-feature models. These are not overwritten by student experiments.
rf_reference_mae = mean_absolute_error(y_test, rf_pred)
hgb_reference_mae = mean_absolute_error(y_test, hgb_pred)
rf_reference_r2 = r2_score(y_test, rf_pred)
hgb_reference_r2 = r2_score(y_test, hgb_pred)
rf_reference_family_mae = family_mae_from_predictions(rf_pred)
hgb_reference_family_mae = family_mae_from_predictions(hgb_pred)

if hgb_reference_mae < rf_reference_mae:
    best_all_feature_model_name = "HistGradientBoostingRegressor"
    best_all_feature_model = hgb_model
    best_all_feature_predictions = hgb_pred
    best_all_feature_mae = hgb_reference_mae
    best_all_feature_family_mae = hgb_reference_family_mae
else:
    best_all_feature_model_name = "RandomForestRegressor"
    best_all_feature_model = rf_model
    best_all_feature_predictions = rf_pred
    best_all_feature_mae = rf_reference_mae
    best_all_feature_family_mae = rf_reference_family_mae

student_experiments = []
student_experiment_models = []

print("Interactive challenge helpers are ready.")
print(f"All-feature random forest MAE: {rf_reference_mae:.3f} eV")
print(f"All-feature gradient boosting MAE: {hgb_reference_mae:.3f} eV")
print(f"Best previous all-feature model: {best_all_feature_model_name}")

In [ ]:
def run_interactive_challenge(
    feature_strategy="Chemistry groups",
    feature_groups=("Electronegativity", "Valence electrons"),
    k_best=25,
    model_type="Random forest",
    max_depth=0,
    n_estimators_or_iterations=150,
    learning_rate=0.05,
    target_family="overall",
):
    """Train and evaluate a student-selected feature/model experiment."""
    all_feature_columns = list(feature_cols)

    # Choose feature columns using the student's selected strategy.
    if feature_strategy == "All features":
        selected_columns = all_feature_columns
        selection_description = "All features"
    elif feature_strategy == "Automatic top-k":
        selected_columns = select_automatic_top_k_features(k_best, all_feature_columns)
        selection_description = f"Automatic top-{len(selected_columns)} features"
    else:
        selected_columns = get_feature_columns_for_groups(feature_groups, all_feature_columns)
        selection_description = ", ".join(feature_groups) if feature_groups else "No groups selected"

    if not selected_columns:
        print("Warning: no features were selected. Choose at least one feature group or use All features.")
        return None

    # Interpret 0 as no maximum depth, which is easier to expose in a widget than Python None.
    depth_value = None if max_depth == 0 else int(max_depth)
    n_steps = int(n_estimators_or_iterations)

    # Keep the train/test split fixed so that different student experiments can be compared fairly.
    X_train_selected = X_train[selected_columns]
    X_test_selected = X_test[selected_columns]

    if model_type == "Random forest":
        model = RandomForestRegressor(
            n_estimators=n_steps,
            max_depth=depth_value,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            max_features="sqrt",
        )
        model_settings = f"n_estimators={n_steps}, max_depth={depth_value}"
    else:
        model = HistGradientBoostingRegressor(
            max_iter=n_steps,
            learning_rate=float(learning_rate),
            max_depth=depth_value,
            l2_regularization=0.01,
            random_state=RANDOM_STATE,
        )
        model_settings = f"max_iter={n_steps}, learning_rate={learning_rate:.3f}, max_depth={depth_value}"

    model.fit(X_train_selected, y_train)
    predictions = model.predict(X_test_selected)

    overall_mae = mean_absolute_error(y_test, predictions)
    overall_r2 = r2_score(y_test, predictions)

    result_table = meta_test.copy().reset_index(drop=True)
    result_table["actual_gap_eV"] = y_test.to_numpy()
    result_table["predicted_gap_eV"] = predictions
    result_table["signed_error_eV"] = result_table["predicted_gap_eV"] - result_table["actual_gap_eV"]
    result_table["absolute_error_eV"] = result_table["signed_error_eV"].abs()
    result_table["chemical_family"] = result_table["composition"].apply(simple_chemical_family)

    family_summary = (
        result_table.groupby("chemical_family")
        .agg(
            MAE_eV=("absolute_error_eV", "mean"),
            count=("absolute_error_eV", "size"),
        )
        .sort_values("MAE_eV")
    )

    if target_family == "overall":
        target_family_mae = np.nan
        previous_family_mae = np.nan
    elif target_family in family_summary.index:
        target_family_mae = family_summary.loc[target_family, "MAE_eV"]
        previous_family_mae = best_all_feature_family_mae.get(target_family, np.nan)
    else:
        print(f"Warning: the selected family '{target_family}' has no test examples.")
        target_family_mae = np.nan
        previous_family_mae = np.nan

    print("Feature strategy:", feature_strategy)
    print("Selected feature groups or setting:", selection_description)
    print("Model type:", model_type)
    print("Model settings:", model_settings)
    print("Number of selected features:", len(selected_columns))
    print(f"Overall MAE: {overall_mae:.3f} eV")
    print(f"Overall R2: {overall_r2:.3f}")
    print("Selected family:", target_family)
    if target_family != "overall":
        print(f"Selected family MAE: {target_family_mae:.3f} eV")
        print(f"Best previous all-feature model family MAE: {previous_family_mae:.3f} eV")

    comparison = pd.DataFrame(
        [
            {"model": "All-feature random forest", "MAE_eV": rf_reference_mae, "R2": rf_reference_r2},
            {"model": "All-feature gradient boosting", "MAE_eV": hgb_reference_mae, "R2": hgb_reference_r2},
            {"model": "Current experiment", "MAE_eV": overall_mae, "R2": overall_r2},
        ]
    )
    display(comparison.round({"MAE_eV": 3, "R2": 3}))

    print(f"Feature efficiency = selected-feature MAE {overall_mae:.3f} eV compared with all-feature random forest MAE {rf_reference_mae:.3f} eV")
    if len(selected_columns) < len(all_feature_columns) and overall_mae < rf_reference_mae:
        print("This model used fewer features and improved MAE.")
    elif len(selected_columns) < len(all_feature_columns) and overall_mae > rf_reference_mae:
        print("This model used fewer features but worsened MAE.")
    else:
        print("This model used many features or performed similarly to the full model.")

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.scatter(y_test, predictions, alpha=0.65, s=35, color="#4C78A8", edgecolor="white", linewidth=0.4)
    min_gap = min(y_test.min(), predictions.min())
    max_gap = max(y_test.max(), predictions.max())
    ax.plot([min_gap, max_gap], [min_gap, max_gap], color="#F58518", linestyle="--")
    ax.set_xlabel("Experimental band gap (eV)")
    ax.set_ylabel("Predicted band gap (eV)")
    ax.set_title("Current experiment: predicted vs. actual")
    ax.set_aspect("equal", adjustable="box")
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(family_summary.index, family_summary["MAE_eV"], color="#B279A2")
    ax.set_ylabel("MAE (eV)")
    ax.set_title("Current experiment: MAE by chemical family")
    ax.tick_params(axis="x", rotation=30)
    plt.show()

    print("10 largest absolute errors:")
    display(
        result_table[["formula", "chemical_family", "actual_gap_eV", "predicted_gap_eV", "absolute_error_eV", "signed_error_eV"]]
        .sort_values("absolute_error_eV", ascending=False)
        .head(10)
        .round(3)
    )

    experiment_number = len(student_experiments) + 1
    experiment_record = {
        "experiment": experiment_number,
        "timestamp": datetime.now().strftime("%H:%M:%S"),
        "feature_strategy": feature_strategy,
        "selected_feature_groups": selection_description,
        "k_best": int(k_best) if feature_strategy == "Automatic top-k" else np.nan,
        "model_type": model_type,
        "model_settings": model_settings,
        "number_of_features": len(selected_columns),
        "overall_test_MAE_eV": overall_mae,
        "overall_test_R2": overall_r2,
        "target_family": target_family,
        "target_family_MAE_eV": target_family_mae,
    }
    student_experiments.append(experiment_record)
    student_experiment_models.append(
        {
            "experiment": experiment_number,
            "model": model,
            "feature_columns": selected_columns,
            "overall_test_MAE_eV": overall_mae,
            "target_family": target_family,
            "target_family_MAE_eV": target_family_mae,
        }
    )

    experiment_log = pd.DataFrame(student_experiments).sort_values("overall_test_MAE_eV")
    print("Experiment log sorted by overall MAE:")
    display(experiment_log.round({"overall_test_MAE_eV": 3, "overall_test_R2": 3, "target_family_MAE_eV": 3}))

    print("Top 5 experiments by overall MAE:")
    display(experiment_log.head(5).round({"overall_test_MAE_eV": 3, "overall_test_R2": 3, "target_family_MAE_eV": 3}))

    family_log = experiment_log.dropna(subset=["target_family_MAE_eV"])
    if not family_log.empty:
        print("Top 5 experiments by selected target-family MAE:")
        display(family_log.sort_values("target_family_MAE_eV").head(5).round({"overall_test_MAE_eV": 3, "overall_test_R2": 3, "target_family_MAE_eV": 3}))

    return experiment_record


if WIDGETS_AVAILABLE:
    feature_group_widget = widgets.SelectMultiple(
        options=list(FEATURE_GROUP_KEYWORDS.keys()),
        value=("Electronegativity", "Valence electrons"),
        description="Groups",
        rows=7,
    )
    depth_widget = widgets.SelectionSlider(
        options=[("None", 0)] + [(str(depth), depth) for depth in range(2, 21)],
        value=0,
        description="max_depth",
        continuous_update=False,
    )

    interact_manual(
        run_interactive_challenge,
        feature_strategy=widgets.Dropdown(
            options=["Chemistry groups", "Automatic top-k", "All features"],
            value="Chemistry groups",
            description="Feature strategy",
        ),
        feature_groups=feature_group_widget,
        k_best=widgets.Dropdown(options=[10, 25, 50, 100], value=25, description="k_best"),
        model_type=widgets.Dropdown(options=["Random forest", "Gradient boosting"], value="Random forest", description="Model"),
        max_depth=depth_widget,
        n_estimators_or_iterations=widgets.IntSlider(value=150, min=50, max=300, step=50, description="Trees/iters"),
        learning_rate=widgets.FloatSlider(value=0.05, min=0.01, max=0.30, step=0.01, readout_format=".2f", description="Learn rate"),
        target_family=widgets.Dropdown(
            options=["overall", "oxide", "halide", "chalcogenide", "pnictide", "elemental", "other"],
            value="overall",
            description="Family",
        ),
    )
else:
    print("Widgets are unavailable. Example manual call:")
    print("run_interactive_challenge(feature_strategy='All features', model_type='Gradient boosting')")

### Mystery formula tester

Use this optional tester to predict a band gap for a formula you type in, such as `Si`, `GaAs`, `CdTe`, `PbS`, `TiO2`, or `CsPbBr3`.

The tester uses the best model from your interactive experiment log if you have run at least one experiment. Otherwise, it uses the default all-feature gradient boosting model.

For quantum dots and nanocrystals, formula-only predictions are especially limited because the optical gap can change with size, surface chemistry, dielectric environment, and processing.

In [ ]:
def get_best_available_challenge_model():
    """Return the best student model if available, otherwise the default all-feature gradient boosting model."""
    if student_experiment_models:
        best_record = min(student_experiment_models, key=lambda record: record["overall_test_MAE_eV"])
        return best_record["model"], best_record["feature_columns"], f"student experiment {best_record['experiment']}"
    return hgb_model, list(feature_cols), "default all-feature gradient boosting model"


def predict_mystery_formula(formula="Si"):
    """Predict a formula-only band gap for a student-entered composition."""
    formula = str(formula).strip()
    if not formula:
        print("Please enter a chemical formula.")
        return None

    try:
        composition = Composition(formula)
    except Exception as exc:
        print(f"Could not parse formula '{formula}'. Check capitalization and stoichiometry.")
        print(exc)
        return None

    model, expected_columns, model_label = get_best_available_challenge_model()

    formula_data = pd.DataFrame({"formula": [composition.reduced_formula], "composition": [composition]})
    formula_features = featurizer.featurize_dataframe(
        formula_data,
        col_id="composition",
        ignore_errors=True,
    )

    X_formula = formula_features.reindex(columns=expected_columns).replace([np.inf, -np.inf], np.nan)

    if X_formula.isna().any(axis=None):
        print("Some required feature values are missing for this formula, so the prediction may not be reliable.")
        print("Try another formula or use an all-feature model that can handle the available descriptors.")
        return None

    predicted_gap = float(model.predict(X_formula)[0])

    print(f"Formula: {formula}")
    print(f"Reduced formula: {composition.reduced_formula}")
    print(f"Model used: {model_label}")
    print(f"Predicted band gap: {predicted_gap:.2f} eV")
    print("Warning: this formula-only prediction does not include structure, polymorph, defects, particle size, surfaces, ligands, or synthesis history.")

    return predicted_gap


if WIDGETS_AVAILABLE:
    interact_manual(
        predict_mystery_formula,
        formula=widgets.Text(value="Si", description="Formula"),
    )
else:
    print("Widgets are unavailable. Example manual call:")
    print("predict_mystery_formula('Si')")

### Questions

1. What feature groups did you expect to matter most before running the model?
2. Which feature groups gave the lowest overall MAE?
3. Did the best overall model also perform best for your chosen chemical family?
4. Did a reduced-feature model perform nearly as well as the all-feature model?
5. Which materials remained difficult to predict?
6. Did your chemistry-guided feature selection beat automatic top-k feature selection?
7. Did automatic feature selection choose features that make chemical sense?
8. What does this teach you about the difference between predictive accuracy and physical understanding?
9. Why is formula-only prediction especially limited for quantum dots or nanocrystals?

## 13. Permutation Importance: A More Careful Feature-Importance Check

### Why use permutation importance?

Built-in tree feature importances can be useful, but they can sometimes be misleading. Permutation importance asks a different question: how much worse does the model become if we randomly shuffle one feature? If shuffling a feature strongly worsens performance, the model was relying on that feature.

Feature importance should still not automatically be interpreted as causation. A feature may matter because it is correlated with other chemical or structural information.

In [ ]:
from sklearn.inspection import permutation_importance

# Choose the better tree-based model according to test MAE.
rf_mae = mean_absolute_error(y_test, rf_pred)
hgb_mae = mean_absolute_error(y_test, hgb_pred)

if hgb_mae < rf_mae:
    best_tree_model = hgb_model
    best_tree_model_name = "HistGradientBoostingRegressor"
else:
    best_tree_model = rf_model
    best_tree_model_name = "RandomForestRegressor"

print(f"Best tree-based model by test MAE: {best_tree_model_name}")

# Use a test-set sample for speed if the test set is large.
# This keeps the extension practical in Colab and classroom settings.
perm_sample_size = min(600, len(X_test))
X_perm = X_test.sample(n=perm_sample_size, random_state=RANDOM_STATE)
y_perm = y_test.loc[X_perm.index]

# Permutation importance measures how much MAE worsens when each feature is shuffled.
perm_result = permutation_importance(
    best_tree_model,
    X_perm,
    y_perm,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_importance_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance_mean_eV": perm_result.importances_mean,
        "importance_std_eV": perm_result.importances_std,
    }
).sort_values("importance_mean_eV", ascending=False)

perm_importance_df.head(15)

In [ ]:
# Plot the top 15 permutation importances.
perm_top15 = perm_importance_df.head(15).sort_values("importance_mean_eV", ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(perm_top15["feature"], perm_top15["importance_mean_eV"], color="#E45756")
ax.set_xlabel("Increase in MAE when feature is shuffled (eV)")
ax.set_title(f"Top 15 permutation importances: {best_tree_model_name}")

perm_plot_path = figure_dir / "top15_permutation_importances.png"
fig.savefig(perm_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {perm_plot_path}")

## 14. What Improved, What Still Failed, and Why?

Use these questions to connect the model-improvement section back to materials chemistry.

1. Did the more flexible model improve MAE?
2. Did it improve all chemical families equally?
3. Which edge cases remained difficult?
4. What missing information would likely help most: structure, processing, defects, dimensionality, or measurement details?
5. Why might composition-only prediction be especially limited for quantum dots and nanocrystals?

### Final Takeaway

A more flexible model can sometimes improve prediction accuracy, but it cannot recover information that is absent from the input data. Composition-based machine learning is useful, but real materials properties also depend on structure, defects, synthesis, measurement conditions, and length scale.

## Glossary

- **Composition**: A representation of which elements are present in a material and their ratios. In this notebook, `pymatgen` stores compositions as `Composition` objects.
- **Feature**: A numerical input used by a machine-learning model. Features in this notebook describe composition-based chemical information.
- **Featurization**: The process of converting a material object, such as a composition, into numerical features.
- **Target variable**: The quantity the model is trying to predict. Here, the target is experimental band gap in eV.
- **Training set**: The part of the dataset used to teach the model patterns between features and target values.
- **Test set**: The held-out part of the dataset used to evaluate predictions on materials the model did not see during training.
- **Baseline model**: A simple reference model. Here, it predicts the average training-set band gap for every material.
- **Random forest**: A model that averages many decision trees to make predictions.
- **Gradient boosting**: A model that builds decision trees sequentially so later trees improve on earlier mistakes.
- **MAE**: Mean absolute error, the average absolute difference between predicted and actual values. In this notebook, it is measured in eV.
- **R2**: A score describing how much variation in the target is explained by the model. Higher is better, with 1 being perfect.
- **Parity plot**: A predicted-vs-actual plot. Perfect predictions lie on the diagonal line.
- **Feature importance**: A model-based estimate of which features were useful for prediction.
- **Permutation importance**: A feature-importance method that measures how much performance worsens when one feature is randomly shuffled.